# FlyOpt Faz 3 Takip Deneyleri (Colab / GPU)

**2026-09-24.** Bu notebook, `notebooks/flyopt_colab_verification_suite.ipynb`'nin
(sağlık kontrolü, alt-graf-tohum sweep, T-duyarlılığı, Pareto null-kontrolü)
DOĞRUDAN DEVAMI — onu tekrar etmiyor, ondan sonraki açık soruları kapatmaya çalışıyor.
Dört bağımsız bölüm, her biri kendi başına çalıştırılabilir (kota biterse en
değerli/ucuz bölümleri önce çalıştır):

- **A. Yedinci fiziksel görev (YENİ): temel taşıma gücü.** Terzaghi taşıma gücü
  denklemi (şerit temel), kullanıcının kendi alanı (geoteknik) — şev stabilitesinin
  kardeş görevi ama tamamen farklı bir arıza mekanizması (döner kayma yerine
  zımbalama/genel-kesme taşıma göçmesi). Eğitim öncesi standart 3 kontrol
  (geçersiz-oran, terim-dengesi, öğretmen-sinyal-kalitesi) BU NOTEBOOK'TAN ÖNCE,
  CPU'da, ayrıca yapıldı — sonuçlar aşağıda. `src/flyopt/benchmarks_bearing.py`
  olarak repoya eklendi.
- **B. Erkek CNS: umut var mı?** δ=0.250 (n=8, FAIL) sonucunun ardında bir
  kalibrasyon/deney-tasarımı sorunu olup olmadığını 5 bağımsız açıdan denetler:
  alt-graf-tohum sağlamlığı, null-model-ailesi taraması, yoğunluk-seyreltme
  kontrolü, DİĞER 6 fiziksel görev, daha büyük alt-graf boyutu. **Not:** "daha
  fazla düşünme süresi" (T-derinliği) hipotezi ZATEN test edildi ve
  DESTEKLENMEDİ (`T_symmetric_robustness.py`, EXPERIMENTS.md 2026-09-21: T=16
  erkek CNS'i δ=0.250'den δ=0.031'e DÜŞÜRDÜ) — burada tekrar edilmiyor.
- **C. 5 geçersiz görev kategorisinin bağlantı-doğrulama düzeltmesi.** Kök neden
  (`select_connected_encode_decode` TAM grafta doğruluyor, `build_subgraph_bfs`
  ALT-GRAFTA garanti etmiyor) izole edilmişti ama düzeltme resmi sonuçlara
  dokunma riski yüzünden bilinçli olarak yapılmamıştı (README §10). Burada:
  önce hatanın büyüklüğü ölçülüyor, sonra resmi şev/kiriş sonuçlarıyla
  REGRESYON testi yapılıyor (sayılar bozulmamalı), sadece o geçerse 5 görev
  düzeltilmiş bağlantı ile yeniden deneniyor.
- **D. Özet + indirme.**

## Kurulum notları
- Bu repo artık **PUBLİK** — `git clone` için token gerekmiyor.
- `data/processed/` altındaki gerekli dosyalar (`.gitignore`'da, git'te YOK,
  ayrıca yüklemen gerekiyor): `adjacency.npz`, `afferent_indices.npy`,
  `efferent_indices.npy` (dişi FlyWire) ve `malecns_adjacency.npz`,
  `malecns_afferent_indices.npy`, `malecns_efferent_indices.npy` (erkek CNS).
- Sonuçlar `flyopt_phase3_results/` altına kaydedilip sonda zip olarak inilir —
  her deney kendi dosyasına ayrı ayrı yazıyor, kota ortasında kesilirse önceki
  hücrelerin sonucu KAYBOLMAZ.


In [ ]:
# --- KURULUM: SECENEK A (onerilen) --- src/ GitHub'dan (repo PUBLIK, token gerekmez),
# data/processed/ sadece senin bilgisayarindan (zip'le yukle).
!git clone -q https://github.com/AutoPyloter/fly_op.git repo
%cd repo

print("Simdi data/processed/ zip'ini yukle (adjacency.npz, afferent_indices.npy,")
print("efferent_indices.npy, malecns_adjacency.npz, malecns_afferent_indices.npy,")
print("malecns_efferent_indices.npy icermeli). PowerShell'de hazirlamak icin:")
print("  cd C:/projeler/fly_op")
print("  Compress-Archive -Path data/processed -DestinationPath flyopt_data_bundle.zip")


In [ ]:
# --- data/processed/ zip yukleme (zip'in ic klasor yapisi ne olursa olsun calisir) ---
from google.colab import files
import zipfile, os, shutil, glob

print("data zip dosyasini sec (adjacency.npz'i icerdigi surece klasor yapisi onemli degil):")
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn) as z:
            z.extractall("_data_extract")
        print(f"{fn} acildi -> _data_extract/")

os.makedirs("data/processed", exist_ok=True)

# adjacency.npz'i (ve kardeslerini) zip'in ICINDE NEREDE OLURSA OLSUN bul --
# PowerShell'in Compress-Archive'i klasor kokunu tutarsiz koyabiliyor, bunu
# tahmin etmek yerine dogrudan ariyoruz.
hit = glob.glob("_data_extract/**/adjacency.npz", recursive=True)
assert hit, "adjacency.npz zip icinde hicbir yerde bulunamadi -- zip'in data/processed/ icerigini (adjacency.npz, afferent_indices.npy, efferent_indices.npy, ...) dogrudan icerdiginden emin ol"
src_dir = os.path.dirname(hit[0])
print(f"kaynak klasor bulundu: {src_dir}")
for f in os.listdir(src_dir):
    s = os.path.join(src_dir, f)
    d = os.path.join("data/processed", f)
    if os.path.isfile(s) and not os.path.exists(d):
        shutil.copy2(s, d)
    elif os.path.isdir(s) and not os.path.exists(d):
        shutil.copytree(s, d)

assert os.path.exists("data/processed/adjacency.npz"), "kopyalama sonrasi hala bulunamadi -- data/processed/ icerigini elle kontrol et"
HAS_MALE = os.path.exists("data/processed/malecns_adjacency.npz")
print("disi FlyWire verisi: OK")
print("erkek CNS verisi bulundu mu:", HAS_MALE, "-- degilse Bolum B atlanir")
print("data/processed/ icerigi:", os.listdir("data/processed"))


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))
sys.path.insert(0, os.path.abspath("scripts"))

!pip install -q networkx pandas scipy matplotlib

import json, time, re, importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import sparse, stats
import networkx as nx

print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

OUT_DIR = "flyopt_phase3_results"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_PROCESSED = "data/processed"

ALL_RESULTS = []  # her deneyin sonuc dict'i buraya toplanir (Bolum D'de export icin)


In [ ]:
from flyopt.substrates.graph_builders import (
    er_null, degree_preserving_rewire, scale_free_null, scale_free_correlated_null,
    community_preserving_rewire, weight_shuffle,
)
from flyopt.variants.fly_proposer_scene import _rays, _teacher_delta
from flyopt.variants.rate_brain import (
    RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode, train,
)

# --- 7 fiziksel gorevin ortak kayit defteri -------------------------------
# slope: DIM=3 (xc,yc,R), digerleri DIM=2. "uniform" = duz rng.uniform(lo,hi)
# (gecersizlik ceza degeriyle objektife gomulu), "valid" = sample_valid_point
# (red-orneklemeyle gecerli nokta arar) -- her gorevin kendi kaynak dosyasindaki
# kurala sadik kalinir.
TASKS = {
    "slope":      dict(module="flyopt.benchmarks_geo",        cost_fn="factor_of_safety", sample="uniform", ray_radius=1.0,   label="sev stabilitesi (resmi bulgu)"),
    "beam":       dict(module="flyopt.benchmarks_structural",  cost_fn="beam_cost",        sample="uniform", ray_radius=0.02,  label="kiris tasarimi (egilme)"),
    "column":     dict(module="flyopt.benchmarks_column",      cost_fn="column_cost",      sample="valid",   ray_radius=0.01,  label="kolon burkulmasi"),
    "deflection": dict(module="flyopt.benchmarks_deflection",  cost_fn="deflection_cost",  sample="valid",   ray_radius=0.01,  label="kiris sehimi"),
    "vessel":     dict(module="flyopt.benchmarks_vessel",      cost_fn="vessel_cost",      sample="valid",   ray_radius=0.001, label="basincli kap"),
    "shear":      dict(module="flyopt.benchmarks_shear",       cost_fn="shear_cost",       sample="valid",   ray_radius=0.002, label="kesme gerilmesi"),
    "bearing":    dict(module="flyopt.benchmarks_bearing",     cost_fn="bearing_cost",     sample="valid",   ray_radius=0.01,  label="temel tasima gucu (YENI, Bolum A)"),
}

def load_task(key):
    spec = TASKS[key]
    mod = importlib.import_module(spec["module"])
    return dict(dim=mod.DIM, cost_fn=getattr(mod, spec["cost_fn"]), geo_fn=mod.random_geometry,
                sample_fn=(mod.sample_valid_point if spec["sample"] == "valid" else None),
                ray_radius=spec["ray_radius"], label=spec["label"])

def cliffs_delta(real, null):
    gt = np.sum(real[:, None] < null[None, :])
    lt = np.sum(real[:, None] > null[None, :])
    return float((gt - lt) / (len(real) * len(null)))

def build_training_set_task(task, n_problems, n_starts, seed):
    rng = np.random.default_rng(seed)
    X, Y = [], []
    dim = task["dim"]
    for _ in range(n_problems):
        geo = task["geo_fn"](rng)
        f = lambda p, geo=geo: task["cost_fn"](p, geo)
        for _ in range(n_starts):
            if task["sample_fn"] is not None:
                x = task["sample_fn"](geo, rng)
            else:
                lo, hi = geo.param_bounds()
                x = rng.uniform(lo, hi)
            fx = f(x)
            rays = _rays(x, f, fx, task["ray_radius"])
            target = _teacher_delta(x, rays, task["ray_radius"])
            norm = np.linalg.norm(target)
            target = target / norm if norm > 1e-8 else np.zeros(dim)
            X.append(rays); Y.append(target)
    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32)

def run_task_on_subgraph(task_key, sub_real, encode_idx, decode_idx, n_seeds, null_fn=None,
                          null_name="degree_preserving", T=8, epochs=300, lr=3e-3, extra=None):
    task = load_task(task_key)
    null_fn = null_fn or (lambda sub, seed: degree_preserving_rewire(sub, seed=seed))
    cfg = RateBrainConfig(dim=task["dim"], n_readout=len(decode_idx), T=T,
                           ray_radius=task["ray_radius"], decode_scale=0.5, train_gain=True)
    real_list, null_list = [], []
    for seed in range(n_seeds):
        X, Y = build_training_set_task(task, 20, 10, seed=9500 + seed)
        X_test, Y_test = build_training_set_task(task, 10, 10, seed=19500 + seed)
        X_test_t = torch.as_tensor(X_test, device=DEVICE)
        Y_test_t = torch.as_tensor(Y_test, device=DEVICE)
        sub_null = null_fn(sub_real, seed)
        for lst, sub in [(real_list, sub_real), (null_list, sub_null)]:
            brain = RateBrain(sub, encode_idx, decode_idx, cfg, seed=seed).to(DEVICE)
            train(brain, X, Y, epochs=epochs, lr=lr)
            with torch.no_grad():
                pred = brain(X_test_t)
                mse = float(((pred - Y_test_t) ** 2).mean().item())
            lst.append(mse)
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    real, null = np.array(real_list), np.array(null_list)
    delta = cliffs_delta(real, null)
    try:
        mw_p = float(stats.mannwhitneyu(real, null, alternative="two-sided").pvalue)
    except ValueError:
        mw_p = float("nan")
    out = dict(task=task_key, label=task["label"], null=null_name, n=n_seeds, T=T,
               n_nodes=int(sub_real.shape[0]), n_edges=int(sub_real.nnz),
               real_median=float(np.median(real)), null_median=float(np.median(null)),
               cliffs_delta=delta, mannwhitney_p=mw_p,
               gate_pass=bool(mw_p < 0.05 and abs(delta) > 0.33),
               real_values=real.tolist(), null_values=null.tolist())
    if extra:
        out.update(extra)
    return out

def run_task_comparison(task_key, base_weights, afferent, efferent, subgraph_seed, n_seeds,
                         null_fn=None, null_name="degree_preserving", T=8, subgraph_size=3000,
                         epochs=300, lr=3e-3, builder=None, extra=None):
    task = load_task(task_key)
    n_rays = 2 * task["dim"]
    builder = builder or build_subgraph_bfs
    encode_full, decode_full = select_connected_encode_decode(
        base_weights, afferent, efferent, n_encode=n_rays, n_decode=30,
        max_hops=6, n_encode_candidates=200, seed=subgraph_seed)
    sub_real, encode_idx, decode_idx, _ = builder(base_weights, encode_full, decode_full, subgraph_size, seed=subgraph_seed)
    result = run_task_on_subgraph(task_key, sub_real, encode_idx, decode_idx, n_seeds, null_fn, null_name, T, epochs, lr, extra)
    result.update(subgraph_seed=subgraph_seed, subgraph_size=subgraph_size)
    return result

def verify_subgraph_connectivity(sub, encode_idx, decode_idx, hop_budget=10):
    # Her encode noronunun decode kumesine ALT-GRAF ICINDE (sadece tam grafta
    # degil) ulasip ulasamadigini BFS ile dogrular -- Bolum C'nin kok-neden
    # testinin ve Bolum B3'un seyreltme-sonrasi kopukluk kontrolunun temeli
    # (README.md Sec. 10, EXPERIMENTS.md 2026-09-23/24).
    n = sub.shape[0]
    coo = sub.tocoo()
    order = np.argsort(coo.col, kind="stable")
    col_sorted, row_sorted = coo.col[order], coo.row[order]
    starts = np.searchsorted(col_sorted, np.arange(n + 1))
    def successors(node):
        return row_sorted[starts[node]:starts[node + 1]]
    decode_set = set(int(d) for d in decode_idx)
    disconnected = []
    for e in np.unique(encode_idx):
        e = int(e)
        seen = {e}; frontier = [e]; reached = False
        for _hop in range(hop_budget):
            nxt = []
            for node in frontier:
                for s in successors(node):
                    s = int(s)
                    if s in decode_set:
                        reached = True
                    if s not in seen:
                        seen.add(s); nxt.append(s)
            if reached or not nxt:
                break
            frontier = nxt
        if not reached:
            disconnected.append(e)
    return disconnected

def compute_real_communities(sub_real):
    coo = sub_real.tocoo()
    G = nx.Graph()
    G.add_nodes_from(range(sub_real.shape[0]))
    edges = set((min(r, c), max(r, c)) for r, c in zip(coo.row.tolist(), coo.col.tolist()) if r != c)
    G.add_edges_from(edges)
    comms = nx.algorithms.community.louvain_communities(G, seed=0, resolution=1.0)
    out = np.zeros(sub_real.shape[0], dtype=np.int64)
    for cid, members in enumerate(comms):
        for m in members:
            out[m] = cid
    return out

def save_result(name, result):
    with open(f"{OUT_DIR}/{name}.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    ALL_RESULTS.append({k: v for k, v in result.items() if k not in ("real_values", "null_values")})
    print(f"[{name}] delta={result['cliffs_delta']:.3f}  p={result['mannwhitney_p']:.4g}  "
          f"gate={'PASS' if result['gate_pass'] else 'FAIL'}  (real_med={result['real_median']:.4f} null_med={result['null_median']:.4f}, n_nodes={result.get('n_nodes')}, n_edges={result.get('n_edges')})")

print("ortak fonksiyonlar hazir --", len(TASKS), "fiziksel gorev kayitli:", list(TASKS))


In [ ]:
female_weights = sparse.load_npz(f"{DATA_PROCESSED}/adjacency.npz")
female_afferent = np.load(f"{DATA_PROCESSED}/afferent_indices.npy")
female_efferent = np.load(f"{DATA_PROCESSED}/efferent_indices.npy")
print(f"disi FlyWire: {female_weights.shape[0]} noron, {female_weights.nnz} kenar")

if HAS_MALE:
    male_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
    male_afferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy")
    male_efferent = np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")
    print(f"erkek CNS: {male_weights.shape[0]} noron, {male_weights.nnz} kenar")


## Bolum A -- Yedinci fiziksel gorev (YENI): temel tasima gucu

**Gorev:** seri/serit temel (strip footing), Terzaghi genel-kesme tasima gucu
denklemi. `x = (B, Df)` -- temel genisligi ve gomulme derinligi. Objektif =
zeminin tasima-gucu kullanim orani (uygulanan basinc / nihai tasima gucu) +
malzeme/kazi maliyeti (`cost_weight * B * Df`) -- kolon/kesme gorevlerinde
ISE YARAYAN ayni "arizanin-kullanim-orani + maliyet" yapisi.

Bu notebook'tan ONCE, GPU gerekmeden (sadece numpy), standart 3 on-egitim
kontrolu zaten yapildi (bkz. `src/flyopt/benchmarks_bearing.py` docstring'i):

```
gecersiz-oran: %0.0  (4000 ornek)
kullanim-terimi payi: medyan %36 (p25=%18, p75=%61)
ogretmen-sinyal kalitesi (radius=0.01): kosinus-benzerligi=1.0000 +/- 0.0000 (n=500-800)
```

Yani kalibrasyon zaten temiz -- asagidaki hucre bunu bu ortamda TEKRAR
dogrular (hizli, CPU), sonra dogrudan n=8 pilotuna geciyor.


In [ ]:
# hizli yeniden-dogrulama (CPU, GPU beklemeden) -- kolon/kesme'de oldugu gibi
# egitimden ONCE calistiriliyor
from flyopt.benchmarks_bearing import DIM as BEARING_DIM, bearing_cost, random_geometry as bearing_geo, sample_valid_point as bearing_sample

rng = np.random.default_rng(0)
shares, invalid = [], 0
for _ in range(2000):
    geo = bearing_geo(rng)
    x = bearing_sample(geo, rng)
    total = bearing_cost(x, geo)
    if total >= 1e10:
        invalid += 1
        continue
    # utilization payi (toplam maliyetin ne kadari ariza-kullanim teriminden geliyor)
    from flyopt.benchmarks_bearing import _q_ult
    util = (geo.load / x[0]) / _q_ult(x[0], x[1], geo.cohesion, geo.friction_deg, geo.unit_weight)
    shares.append(util / total)
shares = np.array(shares)
print(f"gecersiz-oran: %{100*invalid/2000:.1f}")
print(f"kullanim-terimi payi: medyan %{100*np.median(shares):.1f} (p25=%{100*np.percentile(shares,25):.1f} p75=%{100*np.percentile(shares,75):.1f})")
assert invalid / 2000 < 0.05 and 0.10 < np.median(shares) < 0.70, "kalibrasyon beklenenden farkli -- egitime GECME, once incele"
print("kalibrasyon dogrulandi, egitime geciliyor")


In [ ]:
r_bearing = run_task_comparison("bearing", female_weights, female_afferent, female_efferent,
                                  subgraph_seed=9000, n_seeds=8)
save_result("A_bearing_female_n8", r_bearing)
print("\nn=8 esigi astiysa (|delta|>0.33) n=30'a genisletmek icin asagidaki hucreyi ayri calistir.")


### (istege bagli) n=30'a genisletme -- SADECE yukaridaki n=8 esigi astiysa calistir
PROTOCOL.md'nin kurali: |Cliff's delta|>0.33 ise p-degerinden bagimsiz olarak genislet.


In [ ]:
if abs(r_bearing["cliffs_delta"]) > 0.33:
    print(f"esik asildi (delta={r_bearing['cliffs_delta']:.3f}) -- n=30'a genisletiliyor (22 yeni tohum)")
    task = load_task("bearing")
    encode_full, decode_full = select_connected_encode_decode(
        female_weights, female_afferent, female_efferent, n_encode=2*task["dim"], n_decode=30,
        max_hops=6, n_encode_candidates=200, seed=9000)
    sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(female_weights, encode_full, decode_full, 3000, seed=9000)
    cfg = RateBrainConfig(dim=task["dim"], n_readout=len(decode_idx), T=8, ray_radius=task["ray_radius"], decode_scale=0.5, train_gain=True)
    real_ext, null_ext = list(r_bearing["real_values"]), list(r_bearing["null_values"])
    for seed in range(8, 30):
        X, Y = build_training_set_task(task, 20, 10, seed=9500 + seed)
        X_test, Y_test = build_training_set_task(task, 10, 10, seed=19500 + seed)
        sub_null = degree_preserving_rewire(sub_real, seed=seed)
        for lst, sub in [(real_ext, sub_real), (null_ext, sub_null)]:
            brain = RateBrain(sub, encode_idx, decode_idx, cfg, seed=seed).to(DEVICE)
            train(brain, X, Y, epochs=300, lr=3e-3)
            with torch.no_grad():
                pred = brain(torch.as_tensor(X_test, device=DEVICE))
                mse = float(((pred - torch.as_tensor(Y_test, device=DEVICE)) ** 2).mean().item())
            lst.append(mse)
        print(f"  seed={seed} tamamlandi")
    real30, null30 = np.array(real_ext), np.array(null_ext)
    delta30 = cliffs_delta(real30, null30)
    mw_p30 = float(stats.mannwhitneyu(real30, null30, alternative="two-sided").pvalue)
    r_bearing_30 = dict(task="bearing", label="temel tasima gucu (n=30)", null="degree_preserving", n=30,
                         real_median=float(np.median(real30)), null_median=float(np.median(null30)),
                         cliffs_delta=delta30, mannwhitney_p=mw_p30,
                         gate_pass=bool(mw_p30 < 0.05 and abs(delta30) > 0.33),
                         real_values=real30.tolist(), null_values=null30.tolist())
    save_result("A_bearing_female_n30", r_bearing_30)
else:
    print(f"esik asilmadi (delta={r_bearing['cliffs_delta']:.3f}) -- PROTOCOL.md geregi n=30'a CIKARMA, negatif kaydet")


## Bolum B -- Erkek CNS: umut var mi?

Resmi bulgu (disi FlyWire, sev stabilitesi, degree_preserving null, delta=0.884
n=30) erkek Drosophila CNS connectome'unda tekrarlanmadi (delta=0.250, n=8,
gate FAIL -- EXPERIMENTS.md 2026-09-21). Baglanti-dogrulama kontrolu TEMIZ
cikti (pipeline hatasi yok) ama yapisal bir fark bulundu: erkek CNS alt-grafi
disininkinin YARISINDAN AZ yogunlukta/derecede (0.00853 vs 0.01927, ort. derece
46 vs 102) ama DAHA modular (Louvain Q=0.606 vs 0.357). "Daha fazla dusunme
suresi" (T=16) hipotezi zaten test edildi ve DESTEKLENMEDI (delta 0.250 -> 0.031,
yani KOTULESTI). Asagidaki 5 alt-deney HENUZ test EDILMEMIS acik ucları kapatiyor.


### B1 -- Alt-graf-tohum saglamligi (erkek CNS)
Disi tarafta 21 bagimsiz tohum tarandi (onceki notebook); erkek CNS'te SADECE
seed=9000 denendi. seed=9000 tipik mi, yoksa istisna mi?


In [ ]:
if HAS_MALE:
    MALE_SEEDS = list(range(9000, 9010))  # 10 bagimsiz alt-graf secimi
    b1_results = []
    for s in MALE_SEEDS:
        r = run_task_comparison("slope", male_weights, male_afferent, male_efferent,
                                 subgraph_seed=s, n_seeds=6, extra={"connectome": "male_cns"})
        b1_results.append(r)
        save_result(f"B1_male_subgraphseed_{s}", r)
    df_b1 = pd.DataFrame([{k: v for k, v in r.items() if k not in ("real_values", "null_values")} for r in b1_results])
    df_b1.to_csv(f"{OUT_DIR}/B1_male_subgraphseed_sweep.csv", index=False)
    print(f"\nOzet: {df_b1['gate_pass'].sum()}/{len(df_b1)} tohum gate PASS aldi (referans: disi 9/11 -- onceki notebook)")
    print(f"delta dagilimi: medyan={df_b1['cliffs_delta'].median():.3f} min={df_b1['cliffs_delta'].min():.3f} max={df_b1['cliffs_delta'].max():.3f}")
else:
    print("erkek CNS verisi yok, Bolum B atlaniyor")


In [ ]:
if HAS_MALE:
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ["#2a9d8f" if g else "#e76f51" for g in df_b1["gate_pass"]]
    ax.bar(df_b1["subgraph_seed"].astype(str), df_b1["cliffs_delta"], color=colors)
    ax.axhline(0.33, color="gray", linestyle="--", linewidth=1); ax.axhline(-0.33, color="gray", linestyle="--", linewidth=1)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("alt-graf secim tohumu (erkek CNS)"); ax.set_ylabel("Cliff's delta")
    ax.set_title("Erkek CNS: alt-graf-tohum saglamligi (sev stabilitesi)")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/B1_male_subgraphseed_sweep.png", dpi=140); plt.show()


### B2 -- Null-model-ailesi taramasi (erkek CNS)
Erkek CNS su ana kadar SADECE degree_preserving_rewire'a karsi test edildi.
Daha zayif bir null'a karsi bile bir sinyal var mi (kismi/zayif bir yapisal
avantaj mi, yoksa tam bir yokluk mu)?


In [ ]:
if HAS_MALE:
    encode_full_m, decode_full_m = select_connected_encode_decode(
        male_weights, male_afferent, male_efferent, n_encode=6, n_decode=30,
        max_hops=6, n_encode_candidates=200, seed=9000)
    sub_male_9000, enc_m, dec_m, _ = build_subgraph_bfs(male_weights, encode_full_m, decode_full_m, 3000, seed=9000)
    male_communities = compute_real_communities(sub_male_9000)
    print(f"erkek CNS louvain topluluk sayisi: {male_communities.max()+1}")

    NULL_FAMILY_MALE = {
        "er_null": lambda sub, seed: er_null(sub, seed=seed),
        "scale_free": lambda sub, seed: scale_free_null(sub, seed=seed),
        "scale_free_correlated": lambda sub, seed: scale_free_correlated_null(sub, seed=seed),
        "community_preserving": lambda sub, seed: community_preserving_rewire(sub, male_communities, seed=seed),
        "weight_shuffle": lambda sub, seed: weight_shuffle(sub, seed=seed),
        "degree_preserving": lambda sub, seed: degree_preserving_rewire(sub, seed=seed),
    }
    b2_results = []
    for null_name, null_fn in NULL_FAMILY_MALE.items():
        r = run_task_on_subgraph("slope", sub_male_9000, enc_m, dec_m, n_seeds=8, null_fn=null_fn, null_name=null_name)
        b2_results.append(r)
        save_result(f"B2_male_nullfamily_{null_name}", r)
    df_b2 = pd.DataFrame([{k: v for k, v in r.items() if k not in ("real_values", "null_values")} for r in b2_results])
    df_b2.to_csv(f"{OUT_DIR}/B2_male_nullfamily.csv", index=False)
    print(df_b2[["null", "cliffs_delta", "mannwhitney_p", "gate_pass"]])


### B3 -- Yogunluk-seyreltme kontrolu
Erkek CNS'in bulgusuzlugu SADECE dusuk yogunluktan mi kaynaklaniyor? Disi
alt-grafini erkek CNS'in yogunlugu/ortalama derecesine YAPAY olarak seyreltip
AYNI seyreltilmis-disi-kendi-null'una karsi test ediyoruz. Eger avantaj
seyreltilmis diside de kaybolursa -> yogunluk aciklayici. Kaybolmazsa ->
yogunluk tek basina yeterli degil, "erkeklik"in kendisiyle ilgili baska bir
sey var.


In [ ]:
def dilute_to_avg_degree(sub, target_avg_degree, seed):
    rng = np.random.default_rng(seed)
    coo = sub.tocoo()
    n = sub.shape[0]
    n_edges = len(coo.data)
    target_edges = int(round(target_avg_degree * n))
    if target_edges >= n_edges:
        return sub.copy()
    keep = rng.choice(n_edges, size=target_edges, replace=False)
    return sparse.coo_matrix((coo.data[keep], (coo.row[keep], coo.col[keep])), shape=sub.shape).tocsr()

if HAS_MALE:
    male_avg_degree = sub_male_9000.nnz / sub_male_9000.shape[0]
    encode_full_f, decode_full_f = select_connected_encode_decode(
        female_weights, female_afferent, female_efferent, n_encode=6, n_decode=30,
        max_hops=6, n_encode_candidates=200, seed=9000)
    sub_female_9000, enc_f, dec_f, _ = build_subgraph_bfs(female_weights, encode_full_f, decode_full_f, 3000, seed=9000)
    female_avg_degree = sub_female_9000.nnz / sub_female_9000.shape[0]
    print(f"disi ort. derece={female_avg_degree:.2f}  erkek ort. derece={male_avg_degree:.2f}")

    sub_diluted = dilute_to_avg_degree(sub_female_9000, male_avg_degree, seed=0)
    bad = verify_subgraph_connectivity(sub_diluted, enc_f, dec_f)
    print(f"seyreltme sonrasi baglantisiz encode noron sayisi: {len(bad)}/{len(np.unique(enc_f))}")
    if bad:
        print("UYARI: seyreltme bazi encode noronlarini kopardi -- sonuc temkinli yorumlanmali")

    r_diluted = run_task_on_subgraph("slope", sub_diluted, enc_f, dec_f, n_seeds=8,
                                      extra={"connectome": "female_diluted_to_male_density"})
    save_result("B3_female_diluted", r_diluted)
    print(f"\nkarsilastirma: disi tam yogunluk delta=0.884 (n=30, resmi) | "
          f"disi SEYRELTILMIS (erkek yoguluguna) delta={r_diluted['cliffs_delta']:.3f} (n=8) | "
          f"erkek CNS (gercek) delta=0.250 (n=8, referans)")


### B4 -- Erkek CNS'te DIGER 6 fiziksel gorev
Su ana kadar erkek CNS'te SADECE sev stabilitesi denendi. Belki erkek CNS'in
yapisi (daha seyrek AMA daha modular) baska bir gorevin mekanizmasina daha
iyi uyuyordur.


In [ ]:
if HAS_MALE:
    b4_results = []
    for task_key in TASKS:
        if task_key == "bearing":
            continue  # henuz female'de bile n=30'a cikip cikmadigi belli olmayan cok yeni gorev -- once A'nin sonucunu bekle
        r = run_task_comparison(task_key, male_weights, male_afferent, male_efferent,
                                 subgraph_seed=9000, n_seeds=8, extra={"connectome": "male_cns"})
        b4_results.append(r)
        save_result(f"B4_male_task_{task_key}", r)
    df_b4 = pd.DataFrame([{k: v for k, v in r.items() if k not in ("real_values", "null_values")} for r in b4_results])
    df_b4.to_csv(f"{OUT_DIR}/B4_male_all_tasks.csv", index=False)
    print(df_b4[["task", "label", "cliffs_delta", "mannwhitney_p", "gate_pass"]])


### B5 -- Erkek CNS'te daha buyuk alt-graf boyutu
Disi tarafta avantaj 1000-10000 norondan (10x araligi) tum boyutlarda saglam
cikmisti. Erkek CNS'in ham grafi ASLINDA disininkinden BUYUK (166.700 vs
139.255 noron) -- daha buyuk bir alt-graf, erkek CNS'in mutlak kenar sayisini
disininkine yaklastirabilir. Bu, B3'un yogunluk-seyreltme kontrolunun ters
yonden (seyreltmek yerine buyutmek) testi.


In [ ]:
if HAS_MALE:
    b5_results = []
    for size in [3000, 6000, 10000]:
        r = run_task_comparison("slope", male_weights, male_afferent, male_efferent,
                                 subgraph_seed=9000, n_seeds=6, subgraph_size=size,
                                 extra={"connectome": "male_cns"})
        b5_results.append(r)
        save_result(f"B5_male_subgraphsize_{size}", r)
    df_b5 = pd.DataFrame([{k: v for k, v in r.items() if k not in ("real_values", "null_values")} for r in b5_results])
    df_b5.to_csv(f"{OUT_DIR}/B5_male_subgraph_size.csv", index=False)
    print(df_b5[["subgraph_size", "n_nodes", "n_edges", "cliffs_delta", "mannwhitney_p", "gate_pass"]])
    print("\nreferans (disi, ayni boyutlar, onceki oturum): delta=1.00/1.00/0.84/0.81 (1000/3000/6000/10000)")


## Bolum C -- 5 gecersiz gorevin baglanti-dogrulama duzeltmesi

**Kok neden** (README.md Sec. 10, EXPERIMENTS.md 2026-09-23/24): `select_connected_encode_decode`
her encode/decode noronunu TAM grafta (139k noron) dogruluyor; ama `build_subgraph_bfs`
ALT-GRAFI (3000 noron) encode+decode KUMESININ BIRLESIMINDEN disari BFS ile
kuruyor -- bu, baska encode/decode noronlarinin komsuluklari bir encode
noronunun kendi yolunu 3000'lik butceden disari itmesi durumunda, o encode
noronunun ALT-GRAF ICINDE decode kumesinden KOPUK kalmasina izin verebiliyor
(tam grafta baglantili olmasina ragmen). Bu, 2026-09-23/24 gece bataryasindaki
5 gorevin (zamansal entegrasyon, calisma bellegi, anomali tespiti, kaotik
zaman serisi, carpisma-zamani) hicbir sey ogrenememesinin (sabit cikti)
supheli nedeni.

**Duzeltme:** `build_subgraph_bfs_verified` -- normal `build_subgraph_bfs`'i
cagirir, HER encode noronunun decode kumesine ALT-GRAF ICINDE gercekten
ulasip ulasmadigini BFS ile dogrular; kopuk varsa alt-graf buyuklugunu
1.5x artirip (ayni tohum+deneme sayisi) tekrar dener.

**Plan:** once hatanin buyuklugu olculuyor (egitim yok, hizli) -> sonra resmi
sev/kiris sonuclariyla REGRESYON testi (sayilar ONEMLI OLCUDE degismemeli,
degisirse bir sey bozulmus demektir, ILERI GITME) -> sadece regresyon
temizse 5 gorev duzeltilmis baglanti ile yeniden deneniyor.


In [ ]:
def build_subgraph_bfs_verified(weights_csr, encode_idx, decode_idx, n_total, seed,
                                  max_extra_tries=5, hop_budget=10, verbose=True):
    size = n_total
    for attempt in range(max_extra_tries + 1):
        sub, enc, dec, nodes = build_subgraph_bfs(weights_csr, encode_idx, decode_idx, size, seed=seed + attempt)
        bad = verify_subgraph_connectivity(sub, enc, dec, hop_budget=hop_budget)
        if not bad:
            if attempt > 0 and verbose:
                print(f"  [verified] attempt {attempt}: temiz, n_total={size} (seed={seed+attempt})")
            return sub, enc, dec, nodes
        if verbose:
            print(f"  [verified] attempt {attempt}: {len(bad)}/{len(np.unique(enc))} encode noron ALT-GRAF ICINDE kopuk, n_total={size} -- buyutup tekrar deneniyor")
        size = int(size * 1.5)
    raise RuntimeError(f"{max_extra_tries+1} denemede tam baglanti saglanamadi")

print("verify_subgraph_connectivity + build_subgraph_bfs_verified hazir")


### C1 -- Hatanin buyuklugunu olc (egitim yok, hizli)
Resmi sev/kiris ile 5 gecersiz gorevin ORIJINAL (duzeltilmemis) alt-graf
kurulumunda kac encode noron alt-graf icinde kopuk?


In [ ]:
INVALID_TASK_SCRIPTS = {
    "fly_anomaly_detection_multiseed":       ("anomali/sapma tespiti",              2, 3000),
    "fly_delayed_match_sample_multiseed":    ("calisma bellegi (gecikmeli eslesme)", 2, 3000),
    "fly_looming_ttc_multiseed":             ("carpisma-zamani algisi (looming)",    1, 3000),
    "fly_mackey_glass_prediction_multiseed": ("kaotik zaman serisi (Mackey-Glass)",  1, 3000),
    "fly_path_integration_multiseed":        ("zamansal entegrasyon (path integration)", 2, 3000),
}

print("=== C1: baglanti-kopukluk olcumu (SADECE ORIJINAL builder, egitim yok) ===\n")

# referans: resmi sev (n_encode=6) ve kiris (n_encode=4) -- karsilastirma icin
for name, n_encode in [("sev stabilitesi (resmi)", 6), ("kiris tasarimi (resmi)", 4)]:
    ef, df_ = select_connected_encode_decode(female_weights, female_afferent, female_efferent,
                                              n_encode=n_encode, n_decode=30, max_hops=6, n_encode_candidates=200, seed=9000)
    sub, enc, dec, _ = build_subgraph_bfs(female_weights, ef, df_, 3000, seed=9000)
    bad = verify_subgraph_connectivity(sub, enc, dec)
    print(f"{name}: {len(bad)}/{len(np.unique(enc))} encode noron kopuk")

print()
c1_severity = {}
for script, (label, n_encode, subgraph_size) in INVALID_TASK_SCRIPTS.items():
    ef, df_ = select_connected_encode_decode(female_weights, female_afferent, female_efferent,
                                              n_encode=n_encode, n_decode=30, max_hops=6, n_encode_candidates=200, seed=9000)
    sub, enc, dec, _ = build_subgraph_bfs(female_weights, ef, df_, subgraph_size, seed=9000)
    bad = verify_subgraph_connectivity(sub, enc, dec)
    c1_severity[script] = {"label": label, "n_encode": n_encode, "n_bad": len(bad), "n_total_encode": len(np.unique(enc))}
    print(f"{label}: {len(bad)}/{len(np.unique(enc))} encode noron ALT-GRAF ICINDE kopuk")

with open(f"{OUT_DIR}/C1_connectivity_severity.json", "w", encoding="utf-8") as f:
    json.dump(c1_severity, f, indent=2, ensure_ascii=False)


### C2 -- Regresyon testi: resmi sev + kiris, duzeltilmis builder ile
Duzeltilmis builder'la resmi sonuclar (delta=0.884 / 0.613, n=30) buyuk
olcude BOZULMAMALI -- degisirse Bolum C burada DURUYOR, C3'e GECME.


In [ ]:
r_slope_verified = run_task_comparison("slope", female_weights, female_afferent, female_efferent,
                                         subgraph_seed=9000, n_seeds=8, builder=build_subgraph_bfs_verified,
                                         extra={"regression_check": True})
save_result("C2_slope_verified_regression", r_slope_verified)

r_beam_verified = run_task_comparison("beam", female_weights, female_afferent, female_efferent,
                                        subgraph_seed=9000, n_seeds=8, builder=build_subgraph_bfs_verified,
                                        extra={"regression_check": True})
save_result("C2_beam_verified_regression", r_beam_verified)

print(f"\nREFERANS: sev delta=0.884 (n=30) | kiris delta=0.613 (n=30)")
print(f"DUZELTILMIS BUILDER (n=8): sev delta={r_slope_verified['cliffs_delta']:.3f}  kiris delta={r_beam_verified['cliffs_delta']:.3f}")

REGRESSION_OK = (
    r_slope_verified["gate_pass"] and r_beam_verified["gate_pass"]
    and abs(r_slope_verified["cliffs_delta"] - 0.884) < 0.35
    and abs(r_beam_verified["cliffs_delta"] - 0.613) < 0.35
)
print(f"\nREGRESSION_OK = {REGRESSION_OK}")
if not REGRESSION_OK:
    print("UYARI: regresyon esigi asilmadi -- C3'e GECME, sonucu oldugu gibi kaydet ve kullaniciya bildir.")


### C3 -- SADECE regresyon temizse: 5 gorevi duzeltilmis baglanti ile yeniden dene
Her script kendi orijinal ornekleme/ileri-yayilim/egitim mantigini (sample_batch,
forward_sequence, train_sequence) DEGISMEDEN kullaniyor -- SADECE `build_subgraph_bfs`
cagrisi metin-yamasi ile `build_subgraph_bfs_verified` ile degistiriliyor, ve
dosya yollari Colab'a uyarlaniyor. Boylece her gorevin ic mantigini yeniden
yazma riskine girmiyoruz.


In [ ]:
def load_patched_task(script_name, out_dir):
    src = open(f"scripts/{script_name}.py", encoding="utf-8").read()
    src = re.sub(r'"C:/projeler/fly_op/results/([^"]+)"', lambda m: repr(f"{out_dir}/{m.group(1)}"), src)
    ns = {"__name__": "patched_task_module"}
    exec(compile(src, script_name, "exec"), ns)
    return ns

if REGRESSION_OK:
    c3_results = {}
    for script in INVALID_TASK_SCRIPTS:
        print(f"\n=== C3: {script} (baglanti-dogrulamali) ===")
        ns = load_patched_task(script, OUT_DIR)
        ns["DATA_PROCESSED"] = DATA_PROCESSED
        ns["build_subgraph_bfs"] = build_subgraph_bfs_verified
        try:
            ns["main"]()
            summary_path = None
            for fn in os.listdir(OUT_DIR):
                if fn.startswith(script.replace("_multiseed", "")) and fn.endswith("_summary.json"):
                    summary_path = f"{OUT_DIR}/{fn}"
            if summary_path:
                with open(summary_path, encoding="utf-8") as f:
                    c3_results[script] = json.load(f)
        except Exception as e:
            print(f"HATA ({script}): {e}")
            c3_results[script] = {"error": str(e)}
    with open(f"{OUT_DIR}/C3_retried_tasks_summary.json", "w", encoding="utf-8") as f:
        json.dump(c3_results, f, indent=2, ensure_ascii=False)
    for script, r in c3_results.items():
        if "cliffs_delta" in r:
            print(f"{script}: delta={r['cliffs_delta']:.3f} p={r.get('mannwhitney_p', float('nan')):.4g} gate={'PASS' if r.get('gate_pass') else 'FAIL'}")
else:
    print("REGRESSION_OK=False oldugu icin C3 ATLANDI. C2'nin sonuclarini incele.")


## Bolum D -- Ozet + indirme

In [ ]:
df_all = pd.DataFrame(ALL_RESULTS)
df_all.to_csv(f"{OUT_DIR}/D_ALL_RESULTS.csv", index=False)
print(f"toplam {len(df_all)} deney kaydedildi\n")
pd.set_option("display.max_rows", None)
display_cols = [c for c in ["task", "label", "null", "connectome", "n", "subgraph_size", "n_nodes",
                             "n_edges", "cliffs_delta", "mannwhitney_p", "gate_pass"] if c in df_all.columns]
print(df_all[display_cols].to_string(index=False))


In [ ]:
import shutil
zip_path = shutil.make_archive("flyopt_phase3_results", "zip", OUT_DIR)
print("zip hazir:", zip_path)
from google.colab import files
files.download(zip_path)
print("\nBu zip'i FlyOpt oturumuna geri ver -- EXPERIMENTS.md/README.md'ye islenecek.")
